In [ ]:
# Path setup — auto-finds sourceCode_PhiSpaceST/ and loads the `paths` dict.
import sys as _sys, os as _os
from pathlib import Path as _Path
_root = _Path.cwd().resolve()
while not (_root / 'config' / 'paths_template.yml').exists():
    if _root.parent == _root:
        raise FileNotFoundError('Run notebook from inside sourceCode_PhiSpaceST/.')
    _root = _root.parent
_sys.path.insert(0, str(_root / 'scripts' / '00_setup'))
from paths_loader import paths

In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import cell2location

from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

Load trained model. (Consider retrain using filtered features.)

In [ ]:
results_folder = paths["phispace_data_root"] + "/"
# create paths and names to results folders for reference regression and cell2location models
ref_run_name = f'{results_folder}output/Case3/cell2location/reference_signatures'

adata_ref = sc.read_h5ad(
    f'{ref_run_name}/sc.h5ad'
)
mod = cell2location.models.RegressionModel.load(f"{ref_run_name}", adata_ref)

In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:5]

In [ ]:
VisTissueName = 'P17_T1'
run_name = f'{results_folder}output/Case3/cell2location_{VisTissueName}/cell2location_map'

In [ ]:
adata_vis = sc.read_h5ad(f'{results_folder}data/Visium_NSCLC/{VisTissueName}/adata.h5ad')

In [ ]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_vis.var_names, inf_aver.index)
adata_vis = adata_vis[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

# prepare anndata for cell2location model
cell2location.models.Cell2location.setup_anndata(adata=adata_vis)

In [ ]:
# create and train the model
mod = cell2location.models.Cell2location(
    adata_vis, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent
    # hyper-prior which can be estimated from paired histology:
    N_cells_per_location=30,
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
)
mod.view_anndata_setup()

In [ ]:
mod.train(max_epochs=30000,
          # train using full data (batch_size=None)
          batch_size=None,
          # use all data points in training because
          # we need to estimate cell abundance at all locations
          train_size=1,
         )

# plot ELBO loss history during training, removing first 100 epochs from the plot
mod.plot_history(1000)
plt.legend(labels=['full data training']);

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_vis = mod.export_posterior(
    adata_vis, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
)

# Save model
mod.save(f"{run_name}", overwrite=True)

# mod = cell2location.models.Cell2location.load(f"{run_name}", adata_vis)

# Save anndata object with results
adata_file = f"{run_name}/sp.h5ad"
adata_vis.write(adata_file)
adata_file

In [ ]:
adata_vis.obsm['q05_cell_abundance_w_sf'].to_csv(f'{run_name}/q05_abundance.csv')